# Setup

In [ ]:
# Standard library imports
import os
import time
import warnings

# choose the numerical engine BEFORE keras is imported — this line must come first
os.environ['KERAS_BACKEND'] = 'torch'

# Third-party imports: core data
import numpy as np
import pandas as pd

# Third-party imports: deep learning (Keras 3 on the PyTorch backend)
import torch
import keras
from keras.models import Sequential, Model
from keras.layers import (Input, Dense, GRU, LSTM, Reshape, Permute, Flatten,
                          MultiHeadAttention, LayerNormalization,
                          GlobalAveragePooling1D)
from keras.callbacks import EarlyStopping

# Third-party imports: metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Third-party imports: visualization
import matplotlib.pyplot as plt

# Configuration & settings
warnings.simplefilter(action='ignore', category=FutureWarning)

print(f'keras {keras.__version__}, backend: {keras.backend.backend()}')

This set of instructions prepares a Python environment for deep learning tasks, specifically time series analysis using Keras with a PyTorch backend.

Initially, it imports necessary modules from the standard Python library - `os` for interacting with the operating system (like setting environment variables), `time` for time-related operations, and `warnings` to manage warning messages. 

Next, an important step is taken: the Keras backend is explicitly set to 'torch' using `os.environ['KERAS_BACKEND'] = 'torch'`. This must occur *before* importing Keras itself, as it determines which deep learning framework (TensorFlow, Theano, or PyTorch) Keras will utilize for numerical computations.

Following this, several third-party libraries are imported. `numpy` and `pandas` provide powerful tools for numerical computation and data manipulation, respectively.  The core deep learning components come from `torch` and `keras`. Specific layers like `Dense`, `GRU`, `LSTM`, `MultiHeadAttention`, and others are imported directly from `keras.layers` to build neural network models. The `Sequential` and `Model` classes are also imported for defining model architectures.  `EarlyStopping` is brought in from `keras.callbacks` to prevent overfitting during training.

For evaluating the performance of models, functions like `mean_squared_error` and `mean_absolute_error` are imported from `sklearn.metrics`. Finally, `matplotlib.pyplot` is included for creating visualizations. 

The line `warnings.simplefilter(action='ignore', category=FutureWarning)` suppresses warnings related to future changes in the code, keeping the output cleaner. 

Lastly, a print statement displays the Keras version and confirms that PyTorch is being used as the backend. This provides verification of the environment setup.

In [ ]:
# general settings
class CFG:
    data_folder = './data/'
    graph_folder = './graphs/'
    img_dim1 = 16
    img_dim2 = 6
    SEED = 42
    LOOK_BACK = 84            # input window: twelve weeks of history per prediction
    HORIZON = 14              # forecast two weeks ahead in a single shot
    N_ITEMS = 50              # all 50 item-level series of one store, jointly
    TEST_DAYS = 364           # final 52 weeks held out for testing
    VAL_DAYS = 182            # 26 weeks before that used for early stopping
    D_MODEL = 64              # width of a transformer token embedding
    N_HEADS = 4               # parallel attention heads per block
    N_BLOCKS = 2              # stacked encoder blocks
    PATCH_LEN = 14            # patch transformer: one token = two weeks
    RNN_UNITS = 64            # width of the recurrent hidden state
    EPOCHS = 100              # ceiling for every fit (early stopping decides)
    BATCH = 64                # minibatch size
    PATIENCE = 8              # early-stopping patience in epochs

# display style
plt.style.use("seaborn-v0_8")
plt.rcParams["figure.figsize"] = (CFG.img_dim1, CFG.img_dim2)

# reproducibility: seeds Python, NumPy and the deep learning backend in one call
keras.utils.set_random_seed(CFG.SEED)

This section defines configuration settings and styles for a time series forecasting project.

A class named `CFG` is created to hold all these parameters as attributes, making them easily accessible throughout the code. These settings cover data handling, model architecture, training procedures, and visualization preferences. 

Specifically, it establishes paths for storing data (`data_folder`) and generated graphs (`graph_folder`). It also defines dimensions for images used in potential visualizations (`img_dim1`, `img_dim2`), and sets a random seed (`SEED`) for reproducibility.

Key parameters related to the time series itself are defined: `LOOK_BACK` specifies the length of historical data used as input for each prediction (84 days, or twelve weeks), while `HORIZON` indicates how far into the future the model will forecast (14 days, or two weeks). The number of individual item-level time series being analyzed simultaneously is set by `N_ITEMS`.  The lengths of the test (`TEST_DAYS`) and validation (`VAL_DAYS`) sets are also specified.

Several parameters control the architecture of a transformer-based model: `D_MODEL` defines the embedding dimension, `N_HEADS` determines the number of attention heads, `N_BLOCKS` specifies the number of encoder blocks, and `PATCH_LEN` dictates the length of each patch for the transformer.  For recurrent neural network components, `RNN_UNITS` sets the hidden state size.

Training parameters include `EPOCHS` (the maximum number of training iterations), `BATCH` (the batch size), and `PATIENCE` (the number of epochs to wait before early stopping). 

The code then adjusts the default plotting style using `plt.style.use("seaborn-v0_8")` and sets a standard figure size based on the previously defined image dimensions.

Finally, it ensures reproducibility by setting the random seed for Python, NumPy, and the Keras backend all at once using `keras.utils.set_random_seed(CFG.SEED)`. This guarantees that results will be consistent across multiple runs.

In [ ]:
# hardware: use the Apple silicon GPU when there is one, plain CPU otherwise
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'models will be created on: {DEVICE}')

This code snippet determines the computational device - either the Apple Silicon GPU (using Metal Performance Shaders or MPS) or the CPU – and sets it for use in subsequent operations.

It checks if an MPS-enabled backend is available using `torch.backends.mps.is_available()`. If it is, the variable `DEVICE` is assigned the string 'mps', indicating that the Apple Silicon GPU should be used. Otherwise, `DEVICE` is set to 'cpu', meaning computations will be performed on the central processing unit.

A print statement then displays which device has been selected, providing confirmation of where models and tensors will be created and processed. This allows a user to verify whether the code is utilizing the GPU for faster performance if one is available.

# Utils

In [ ]:
def forecast_metrics(y_true, y_pred):
    """Return MAE and RMSE between two aligned arrays, rounded for display."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {'MAE': float(np.round(mae, 3)), 'RMSE': float(np.round(rmse, 3))}

This code defines a function called `forecast_metrics` that calculates and returns common evaluation metrics for time series forecasting models: Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE).

The function takes two arguments, `y_true` and `y_pred`, which represent the actual (true) values and the predicted values, respectively. It assumes these are aligned arrays – meaning corresponding elements in each array represent the same time point or observation. 

Inside the function, it calculates MAE using `mean_absolute_error(y_true, y_pred)` from scikit-learn.  RMSE is computed by first calculating the mean squared error with `mean_squared_error(y_true, y_pred)`, and then taking its square root using `np.sqrt()`.

Finally, it returns a dictionary containing the calculated MAE and RMSE values. Both metrics are rounded to three decimal places using `np.round()` and converted to floating-point numbers using `float()` for cleaner display. This ensures that the output is easily readable and comparable.

In [ ]:
def split_starts(n_days, look_back, horizon, val_start, test_start, test_stride=7):
    """Chronological train/val/test split of forecast origins (first predicted day)."""
    starts = np.arange(look_back, n_days - horizon + 1)
    train = starts[starts + horizon <= val_start]
    val = starts[(starts >= val_start) & (starts + horizon <= test_start)]
    test = starts[starts >= test_start][::test_stride]
    return train, val, test

This code defines a function called `split_starts` that generates indices for creating chronological training, validation, and testing splits for time series forecasting. It focuses on defining the starting points of each forecast origin – essentially, the first day for which a prediction is made.

The function takes several arguments: `n_days`, representing the total number of days in the dataset; `look_back`, the length of the input window used for predictions; `horizon`, the length of the forecast horizon (how many days ahead to predict); `val_start`, the day at which the validation set begins; and `test_start`, the day at which the test set begins.  `test_stride` controls how frequently test forecasts are selected, defaulting to a step of 7 days.

First, it creates an array called `starts` using `np.arange()`. This array contains all possible starting points for forecasts, ranging from `look_back` (because you need enough history to look back) up to `n_days - horizon + 1` (to ensure there’s enough data remaining to make a forecast of the specified `horizon`).

Then, it divides these starting points into three sets: `train`, `val`, and `test`. The `train` set includes all starting points where the end of the input window (`starts + horizon`) is before or equal to `val_start`.  The `val` set contains starting points that fall between `val_start` (inclusive) and `test_start` (inclusive). Finally, the `test` set consists of starting points greater than or equal to `test_start`, sampled with a stride of `test_stride`.

The function returns these three arrays – `train`, `val`, and `test` – which represent the indices for creating the respective splits in the time series data. These indices can then be used to select the appropriate portions of the dataset for training, validation, and testing the forecasting model.

In [ ]:
def make_windows(values, starts, look_back, horizon):
    """Slice a (days, channels) array into (X, y) tensors at the given origins."""
    X = np.stack([values[s - look_back:s] for s in starts])
    y = np.stack([values[s:s + horizon] for s in starts])
    return X.astype('float32'), y.astype('float32')

This code defines a function called `make_windows` that prepares time series data into input-output pairs (X, y) suitable for training a forecasting model. It takes slices of the original time series based on specified starting points.

The function accepts four arguments: `values`, which is a NumPy array representing the time series data (assumed to have shape (days, channels)); `starts`, an array of indices indicating the starting points for each forecast origin; `look_back`, the length of the input window; and `horizon`, the length of the forecast horizon.

It constructs the input features (`X`) by iterating through the `starts` array. For each starting point `s`, it extracts a slice of the `values` array from `s - look_back` to `s`. This slice represents the historical data used as input for predicting the future. These slices are then stacked together using `np.stack()` to create the `X` array, which has shape (number of samples, look_back, channels).

Similarly, it creates the target values (`y`) by extracting slices from `s` to `s + horizon` for each starting point in `starts`. These slices represent the actual future values that the model will try to predict.  These are also stacked using `np.stack()` resulting in a `y` array with shape (number of samples, horizon, channels).

Finally, both `X` and `y` arrays are cast to the `'float32'` data type for compatibility with most deep learning frameworks, and they are returned as a tuple. This function effectively transforms the raw time series data into a format that can be fed directly into a forecasting model.

In [ ]:
# bookkeeping for the running theme of this episode: accuracy vs cost
train_times, param_counts = {}, {}

def fit_timed(name, model, X_tr, y_tr, X_va, y_va, batch=CFG.BATCH):
    """Fit with early stopping; record wall-clock time and parameter count."""
    early_stop = EarlyStopping(monitor='val_loss', patience=CFG.PATIENCE,
                               restore_best_weights=True)
    t0 = time.time()
    history = model.fit(X_tr, y_tr, validation_data=(X_va, y_va),
                        epochs=CFG.EPOCHS, batch_size=batch,
                        callbacks=[early_stop], verbose=0)
    train_times[name] = round(time.time() - t0, 1)
    param_counts[name] = model.count_params()
    print(f'{name}: {len(history.history["loss"])} epochs, {train_times[name]}s, '
          f'{param_counts[name]:,} params, best val loss '
          f'{min(history.history["val_loss"]):.4f}')
    return model

This code defines a function `fit_timed` designed to train a Keras model while tracking and recording the training time and number of parameters in the model. It’s part of an experiment focused on comparing accuracy versus computational cost.

The function takes several arguments: `name`, a string identifying the model being trained; `model`, the Keras model itself; `X_tr` and `y_tr`, the training data (input features and target values); `X_va` and `y_va`, the validation data; and an optional `batch` size, defaulting to the value specified in the `CFG` class.

It initializes an `EarlyStopping` callback to halt training when the validation loss stops improving for a certain number of epochs (`CFG.PATIENCE`), restoring the best weights encountered during training. 

The code then records the start time using `time.time()`. It proceeds to train the model using `model.fit()`, passing in the training data, validation data, maximum number of epochs (`CFG.EPOCHS`), batch size, and the early stopping callback. The `verbose=0` argument suppresses output during training.

After training completes (either by reaching the maximum number of epochs or triggering early stopping), the function calculates the total training time by subtracting the start time from the current time and rounds it to one decimal place. It also determines the number of trainable parameters in the model using `model.count_params()`. 

These values – training time and parameter count – are stored in dictionaries called `train_times` and `param_counts`, respectively, using the provided `name` as the key. A formatted print statement then displays information about the training process, including the number of epochs trained, the training time, the number of parameters (formatted with commas for readability), and the best validation loss achieved during training.

Finally, the function returns the trained model. This allows subsequent evaluation or further use of the model. The primary purpose is to systematically record performance metrics alongside computational cost for comparison between different models or configurations.

In [ ]:
def score_in_units(pred_z, starts, horizon, values, mean, std):
    """Undo standardization, then score against the raw series in original units."""
    pred = pred_z * std + mean
    actual = np.stack([values[s:s + horizon] for s in starts])
    return forecast_metrics(actual.ravel(), pred.ravel())

This code defines a function called `score_in_units` that evaluates the performance of predictions after converting them back to their original scale from a standardized representation.

The function takes five arguments: `pred_z`, the predicted values in standardized form (i.e., zero mean and unit variance); `starts`, an array of indices indicating the starting points for each forecast origin; `horizon`, the length of the forecast horizon; `values`, the original time series data; `mean`, the mean used for standardization; and `std`, the standard deviation used for standardization.

First, it reverses the standardization process by multiplying the standardized predictions (`pred_z`) by the standard deviation (`std`) and adding the mean (`mean`). This converts the predictions back to their original units. The result is stored in the `pred` variable.

Next, it extracts the corresponding actual values from the original time series data (`values`) using the provided starting points (`starts`) and forecast horizon (`horizon`). These actual values are stacked into a NumPy array called `actual`.

Finally, it calls the `forecast_metrics` function (defined previously) to calculate the Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE) between the flattened predicted values (`pred.ravel()`) and the flattened actual values (`actual.ravel()`). The resulting dictionary containing these metrics is returned.

In essence, this function takes standardized predictions, converts them back to their original scale, compares them to the true values in those original units, and provides a performance evaluation using MAE and RMSE. This ensures that the evaluation reflects the real-world impact of the forecasts.

In [ ]:
def plot_test_forecast(pred_z, item, title, alpha=1.0):
    """Overlay one item's stitched weekly forecasts on the observed test year."""
    pred = pred_z[:, :, item - 1] * train_std[item - 1] + train_mean[item - 1]
    days, vals = [], []
    for i, s in enumerate(test_starts):
        days.extend(panel.index[s:s + 7])   # keep the first week of each origin
        vals.extend(pred[i, :7])
    obs = panel[item].iloc[test_start:]
    plt.figure()
    plt.plot(obs.index, obs.values, linewidth=2, label='observed', alpha=alpha)
    plt.plot(days, vals, 'g--', marker='o', markersize=3, linewidth=1.5,
             label='forecast (first week of each origin)')
    plt.title(title)
    plt.legend()
    plt.xlabel('')
    plt.show()

This code defines a function called `plot_test_forecast` that visualizes the forecasted values against the observed data for a single item during the test period. It overlays weekly forecasts originating from different starting points on the actual time series.

The function takes several arguments: `pred_z`, the standardized predictions; `item`, the index of the specific item being plotted (assuming multiple items are being forecast); `title`, a string used as the plot title; and an optional `alpha` value for controlling the transparency of the observed data line.

First, it converts the standardized predictions (`pred_z`) back to their original scale using the item-specific standard deviation (`train_std[item - 1]`) and mean (`train_mean[item - 1]`). It selects only the predictions relevant to the specified `item` from the prediction array.

Next, it initializes empty lists called `days` and `vals`. It then iterates through the `test_starts` array (containing the starting points for test forecasts). Inside the loop, it extends the `days` list with the dates corresponding to the first week of each forecast origin using `panel.index[s:s + 7]`.  It also extends the `vals` list with the forecasted values for that same first week (`pred[i, :7]`).

The actual observed values for the specified item during the test period are extracted from the `panel` DataFrame and stored in the `obs` variable.

Finally, it creates a new plot using `plt.figure()`. It plots the observed data (`obs`) as a solid line with a specified linewidth and transparency (`alpha`).  It then overlays the forecasted values (`vals`) against their corresponding dates (`days`) as a dashed green line with markers. The plot is titled with the provided `title`, includes a legend, removes the x-axis label, and displays the plot using `plt.show()`.

In essence, this function provides a visual comparison between the model’s forecasts (specifically, the first week of each forecast origin) and the actual observed values during the test period for a single item. This allows for a qualitative assessment of the forecasting performance.

# Groundwork

In [ ]:
# load five years of daily unit sales; carve out store 1 as a 50-channel panel
df = pd.read_csv(CFG.data_folder + 'train.csv', parse_dates=['date'])
panel = (df[df['store'] == 1]
         .pivot(index='date', columns='item', values='sales')
         .astype('float32'))
N_DAYS = len(panel)

print(f'{N_DAYS} days x {panel.shape[1]} items, '
      f'{panel.index.min().date()} to {panel.index.max().date()}')
panel.head(3)

This code loads and preprocesses a time series dataset of daily unit sales for multiple items in a single store.

It begins by reading the data from a CSV file named 'train.csv' located in the directory specified by `CFG.data_folder` using `pd.read_csv()`. The `parse_dates=['date']` argument ensures that the 'date' column is parsed as datetime objects.

Next, it filters the DataFrame to include only data from store 1 (`df[df['store'] == 1]`). It then uses the `.pivot()` method to reshape the data into a panel format where dates are the index, items are the columns, and sales values are the entries. This creates a time series for each item in the store. The resulting DataFrame is cast to the `'float32'` data type using `.astype('float32')`.

The number of days in the dataset is determined by calculating the length of the panel's index (`len(panel)`) and stored in the `N_DAYS` variable. 

A print statement then displays information about the dimensions of the panel (number of days and items), as well as the start and end dates of the data. Finally, it prints the first three rows of the panel using `.head(3)` to provide a quick preview of the data structure.

In summary, this code loads sales data, filters it for a specific store, transforms it into a time series panel where each column represents an item’s sales over time, and provides basic information about the dataset's size and date range.

In [ ]:
# a first look at three channels with very different volumes
panel[[1, 15, 28]].plot(subplots=True, figsize=(CFG.img_dim1, CFG.img_dim2 * 1.4),
                        xlabel='')
plt.suptitle('Store 1 daily unit sales, three of the fifty items', y=0.98)
plt.show()

This code generates a plot visualizing the time series data for three specific items from the loaded dataset.

It selects columns 1, 15, and 28 from the `panel` DataFrame (representing three different items). The `.plot(subplots=True)` method creates separate subplots for each item's time series.  The `figsize` argument sets the overall figure size based on the dimensions defined in the `CFG` class, adjusting the height to accommodate all subplots. The x-axis label is removed using `xlabel=''`.

A title is added to the entire figure using `plt.suptitle()`, indicating that the plot shows daily unit sales for three of the fifty items in store 1.  The `y=0.98` argument adjusts the vertical position of the suptitle to prevent overlap with the subplots.

Finally, the plot is displayed using `plt.show()`. This allows a visual inspection of the time series data for these three selected items, potentially revealing differences in their sales volumes and patterns. The intention is likely to demonstrate that different items exhibit varying levels of demand and temporal characteristics.

In [ ]:
# quantify the co-movement: correlation between every pair of channels
corr = np.corrcoef(panel.values.T)
off_diag = corr[np.triu_indices(CFG.N_ITEMS, k=1)]

plt.figure(figsize=(CFG.img_dim2 * 0.9, CFG.img_dim2 * 0.8))
plt.imshow(corr, cmap='viridis', vmin=0, vmax=1)
plt.colorbar(label='Pearson correlation')
plt.title('Correlation matrix of the fifty item-level series')
plt.xlabel('item')
plt.ylabel('item')
plt.show()

print(f'off-diagonal correlations: min {off_diag.min():.2f}, '
      f'mean {off_diag.mean():.2f}, max {off_diag.max():.2f}')

This code calculates and visualizes the correlation between all pairs of item time series in the dataset, providing insights into their co-movement patterns.

First, it computes the Pearson correlation coefficient matrix using `np.corrcoef(panel.values.T)`. The `.values.T` transposes the panel DataFrame so that each column represents a time series for an item, which is required by `np.corrcoef()`.

Next, it extracts the upper triangular portion of the correlation matrix (excluding the diagonal) using `np.triu_indices(CFG.N_ITEMS, k=1)` to avoid redundant comparisons and self-correlations. The resulting values are stored in the `off_diag` array.

It then creates a heatmap visualization of the full correlation matrix using `plt.imshow()`. The `cmap='viridis'` argument specifies the color map, `vmin=0` and `vmax=1` set the minimum and maximum color values to represent the range of correlation coefficients, and a colorbar is added for interpretation.  The plot is titled "Correlation matrix of the fifty item-level series," with labels for the x and y axes indicating items.

Finally, it prints summary statistics of the off-diagonal correlations: the minimum value, the mean value, and the maximum value, formatted to two decimal places. This provides a quantitative overview of the strength and range of co-movement between the item time series.

In essence, this code assesses how strongly different items’ sales patterns are related to each other, which can be useful for understanding dependencies and potential opportunities for joint forecasting or inventory management.

In [ ]:
# chronological boundaries, then per-channel standardization fitted on train only
test_start = N_DAYS - CFG.TEST_DAYS
val_start = test_start - CFG.VAL_DAYS

train_mean = panel.values[:val_start].mean(axis=0)
train_std = panel.values[:val_start].std(axis=0)
z = (panel.values - train_mean) / train_std

print(f'train: {panel.index[0].date()} .. {panel.index[val_start - 1].date()} '
      f'({val_start} days)')
print(f'val:   {panel.index[val_start].date()} .. {panel.index[test_start - 1].date()} '
      f'({CFG.VAL_DAYS} days)')
print(f'test:  {panel.index[test_start].date()} .. {panel.index[-1].date()} '
      f'({CFG.TEST_DAYS} days)')

This code defines the chronological boundaries for training, validation, and testing sets, and then performs standardization of the time series data using statistics calculated only from the training set.

It first calculates the starting indices for the test and validation sets based on the total number of days (`N_DAYS`) and the lengths specified in the `CFG` class (`TEST_DAYS`, `VAL_DAYS`).  `test_start` marks the beginning of the test set, and `val_start` marks the beginning of the validation set.

Next, it calculates the mean and standard deviation for each item (column) in the panel using only the training data (up to `val_start`). This is crucial to prevent information leakage from the validation or test sets into the standardization process. The results are stored in `train_mean` and `train_std`, respectively.

Then, it standardizes the entire dataset by subtracting the training mean (`train_mean`) from each value and dividing by the corresponding training standard deviation (`train_std`). This results in a standardized panel where each item has zero mean and unit variance within the training period. The standardized data is stored in the `z` variable.

Finally, it prints information about the date ranges and lengths of the training, validation, and test sets to confirm the split. These print statements use the index of the `panel` DataFrame to display the actual dates corresponding to the start and end points of each set.

In summary, this code prepares the data for model training by defining clear chronological splits and standardizing the time series using statistics derived solely from the training data, ensuring a robust evaluation process.

In [ ]:
# window the panel for the main task: 84 days in, 14 days out
train_starts, val_starts, test_starts = split_starts(
    N_DAYS, CFG.LOOK_BACK, CFG.HORIZON, val_start, test_start)

X_train, y_train = make_windows(z, train_starts, CFG.LOOK_BACK, CFG.HORIZON)
X_val, y_val = make_windows(z, val_starts, CFG.LOOK_BACK, CFG.HORIZON)
X_test, y_test = make_windows(z, test_starts, CFG.LOOK_BACK, CFG.HORIZON)

print(f'X_train {X_train.shape}, y_train {y_train.shape}')
print(f'X_val   {X_val.shape},  y_val   {y_val.shape}')
print(f'X_test  {X_test.shape},   y_test  {y_test.shape}')

This code prepares the standardized time series data into input-output pairs (windows) for training, validation, and testing a forecasting model using the defined lookback and horizon parameters.

It begins by calling the `split_starts` function to generate arrays of starting indices (`train_starts`, `val_starts`, `test_starts`) for each set based on the total number of days (`N_DAYS`), lookback period (`CFG.LOOK_BACK`), forecast horizon (`CFG.HORIZON`), and previously defined validation and test start dates.

Next, it calls the `make_windows` function three times – once for each dataset (train, val, test). This function takes the standardized data (`z`), the starting indices, lookback period, and horizon as input and returns the corresponding input features (X) and target values (y) in a windowed format.

Finally, it prints the shapes of the resulting `X_train`, `y_train`, `X_val`, `y_val`, `X_test`, and `y_test` arrays to verify that the data has been correctly prepared for model training. The shape will be (number of samples, lookback, number of items) for X and (number of samples, horizon, number of items) for y.

In essence, this code transforms the standardized time series into a format suitable for supervised learning, where each sample consists of a sequence of past values (input window) and the corresponding future values to be predicted (target).

In [ ]:
# helper bound to this task's globals, to keep later cells short
def score_test(pred_z):
    """Score standardized predictions for the 51 test origins, in sales units."""
    return score_in_units(pred_z, test_starts, CFG.HORIZON,
                          panel.values, train_mean, train_std)

# two seasonal naive baselines every model must beat
results = {}

# lag-7: tile the LAST OBSERVED week across the horizon (no peeking at week one's
# actuals to forecast week two)
naive_7 = np.stack([np.tile(z[s - 7:s], (CFG.HORIZON // 7, 1)) for s in test_starts])
results['seasonal naive (lag 7)'] = score_test(naive_7)

naive_364 = np.stack([z[s - 364:s + CFG.HORIZON - 364] for s in test_starts])
results['seasonal naive (lag 364)'] = score_test(naive_364)

results

This code defines a helper function `score_test` and establishes two seasonal naïve baseline models to serve as benchmarks for evaluating the performance of more complex forecasting models.

The `score_test` function simplifies the scoring process by encapsulating the call to `score_in_units`. It takes standardized predictions (`pred_z`) as input, uses the pre-defined `test_starts`, forecast horizon (`CFG.HORIZON`), original panel values, training mean, and training standard deviation to calculate MAE and RMSE in the original sales units.

Next, it initializes a dictionary called `results` to store the performance metrics of different models. 

Two seasonal naïve baseline models are created:

1.  **naive\_7:** This model predicts future values by repeating the last observed week (7 days) across the forecast horizon. It uses `np.tile()` to replicate the last week's sales for each item and stacks these replicated weeks for all test origins. The code ensures that it doesn’t “peek” at the first week of actuals when forecasting subsequent weeks.

2.  **naive\_364:** This model predicts future values by repeating the sales from 364 days ago (approximately one year) across the forecast horizon. It extracts a slice of data representing the previous year's sales for each test origin and stacks them to create the predictions.

The performance of each baseline model is evaluated using `score_test`, and the resulting MAE and RMSE values are stored in the `results` dictionary, keyed by the model name.

Finally, the code displays the `results` dictionary, which contains the evaluation metrics for the two seasonal naïve baselines. These baselines provide a simple point of comparison to assess whether more sophisticated models offer significant improvements in forecasting accuracy.

# Recurrent networks

In [ ]:
def build_rnn(cell, look_back=CFG.LOOK_BACK, n_channels=CFG.N_ITEMS,
              horizon=CFG.HORIZON, units=CFG.RNN_UNITS):
    """Recurrent encoder + linear head for direct multivariate forecasting."""
    with keras.device(DEVICE):
        model = Sequential([
            Input(shape=(look_back, n_channels)),
            cell(units),
            Dense(horizon * n_channels),
            Reshape((horizon, n_channels)),
        ])
    model.compile(loss='mse', optimizer=keras.optimizers.Adam(1e-3))
    return model

This code defines a function `build_rnn` that constructs a recurrent neural network (RNN) model for multivariate time series forecasting.

The function takes several arguments: `cell`, which specifies the type of RNN cell to use (e.g., GRU, LSTM); `look_back`, the length of the input sequence; `n_channels`, the number of features or items being forecast; `horizon`, the length of the forecast horizon; and `units`, the number of units in the recurrent layer.

Inside the function, it creates a sequential Keras model using `Sequential()`. The model consists of the following layers:

1.  **Input Layer:** An `Input` layer defines the expected shape of the input data as (look\_back, n\_channels).
2.  **Recurrent Layer:** A recurrent layer is created using the provided `cell` (e.g., GRU or LSTM) with the specified number of units. This layer processes the sequential input data and captures temporal dependencies.
3.  **Dense Layer:** A fully connected (`Dense`) layer maps the output of the recurrent layer to a vector of size horizon \* n\_channels, representing the predicted values for each item over the forecast horizon.
4.  **Reshape Layer:** A `Reshape` layer transforms the output of the dense layer into a tensor with shape (horizon, n\_channels), which represents the forecasted time series for each item.

The model is then compiled using the mean squared error (`mse`) loss function and the Adam optimizer with a learning rate of 1e-3. The `keras.device(DEVICE)` context manager ensures that the model is created on the specified device (CPU or GPU).

Finally, the function returns the constructed Keras model. This allows for easy instantiation and training of RNN models with different configurations.

In [ ]:
keras.utils.set_random_seed(CFG.SEED)
gru_model = fit_timed('GRU', build_rnn(GRU), X_train, y_train, X_val, y_val)

gru_pred = gru_model.predict(X_test, verbose=0)
results['GRU'] = score_test(gru_pred)
results['GRU']

This code trains a Gated Recurrent Unit (GRU) model and evaluates its performance on the test set.

First, it sets the random seed for Keras using `keras.utils.set_random_seed(CFG.SEED)` to ensure reproducibility of results.

Then, it calls the `build_rnn` function with `GRU` as the recurrent cell type to create a GRU model. The resulting model is then trained using the `fit_timed` function, passing in the training data (`X_train`, `y_train`), validation data (`X_val`, `y_val`), and a name ('GRU') for tracking purposes.  The trained GRU model is assigned to the variable `gru_model`.

Next, it uses the trained `gru_model` to make predictions on the test data (`X_test`) using `gru_model.predict()`. The `verbose=0` argument suppresses output during prediction. The predicted values are stored in the `gru_pred` variable.

The performance of the GRU model is then evaluated using the `score_test` function, which calculates MAE and RMSE on the test set. The resulting metrics are stored in the `results` dictionary with the key 'GRU'.

Finally, it prints the contents of the `results['GRU']` dictionary, displaying the calculated MAE and RMSE for the GRU model. This provides a quantitative assessment of the model's forecasting accuracy on the test data.

In [ ]:
keras.utils.set_random_seed(CFG.SEED)
lstm_model = fit_timed('LSTM', build_rnn(LSTM), X_train, y_train, X_val, y_val)

results['LSTM'] = score_test(lstm_model.predict(X_test, verbose=0))
results['LSTM']

This code trains a Long Short-Term Memory (LSTM) model and evaluates its performance on the test set, mirroring the process done for the GRU model.

It begins by setting the random seed using `keras.utils.set_random_seed(CFG.SEED)` to ensure reproducibility.

Then, it constructs an LSTM model by calling the `build_rnn` function with `LSTM` as the recurrent cell type. This model is subsequently trained using the `fit_timed` function, utilizing the training data (`X_train`, `y_train`), validation data (`X_val`, `y_val`), and assigning it the name 'LSTM' for tracking purposes. The resulting trained LSTM model is stored in the variable `lstm_model`.

Next, predictions are generated on the test data (`X_test`) using the trained `lstm_model` via `lstm_model.predict()`, with output suppressed by setting `verbose=0`. These predictions are saved to the `lstm_pred` variable (though this variable isn’t explicitly assigned in the provided code, it's implied).

The performance of the LSTM model is then assessed using the `score_test` function, which calculates MAE and RMSE on the test set. The resulting metrics are stored in the `results` dictionary under the key 'LSTM'.

Finally, the contents of `results['LSTM']` are printed, displaying the calculated MAE and RMSE for the LSTM model, providing a quantitative measure of its forecasting accuracy. This allows for comparison with the GRU model’s performance.

In [ ]:
plot_test_forecast(gru_pred, item=15,
                   title='GRU, direct 14-day forecasts over the test year (item 15)')

This code generates a visualization of the forecasted values from the GRU model against the observed data for a specific item.

It calls the `plot_test_forecast` function with the following arguments:

*   `gru_pred`: The predicted values generated by the trained GRU model.
*   `item=15`: Specifies that the plot should focus on item number 15.
*   `title='GRU, direct 14-day forecasts over the test year (item 15)'`: Sets the title of the plot to clearly indicate that it displays the GRU model’s 14-day forecasts for item 15 over the test period.

The `plot_test_forecast` function then overlays the forecasted values (specifically, the first week of each forecast origin) on top of the actual observed sales data for item 15 during the test period. The resulting plot provides a visual comparison between the model’s predictions and the true values, allowing for qualitative assessment of its performance for that particular item.

# Attention from first principles

In [ ]:
def softmax(a):
    """Row-wise softmax: turn scores into weights that are positive and sum to 1."""
    e = np.exp(a - a.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

window = z[100:128, 0]                    # 28 days of item 1, standardized

Q = K = V = window.reshape(-1, 1)         # each day is its own 1-number "token"
scores = Q @ K.T / np.sqrt(Q.shape[1])    # how relevant is day j to day i?
weights = softmax(scores)                 # each row: a probability distribution
attended = weights @ V                    # each day becomes a weighted mix of all days

print(f'weights: {weights.shape}, each row sums to {weights.sum(axis=1)[0]:.4f}')

This code demonstrates a simplified self-attention mechanism, illustrating how it can be used to weigh different parts of an input sequence based on their relevance to each other.

First, a `softmax` function is defined. This function takes an array `a` as input and applies the softmax operation row-wise. It subtracts the maximum value along each row from all elements in that row (to prevent numerical instability), exponentiates the result, and then normalizes by dividing each element by the sum of the exponentiated values in its row. This ensures that the output consists of positive weights that sum to 1 for each row, representing a probability distribution.

Next, a window of standardized data (`z`) is extracted, specifically 28 days of sales data for item 1 (column 0). This window is assigned to the variable `window`.

The code then prepares the input for the self-attention mechanism by reshaping the `window` into matrices `Q`, `K`, and `V`. Each day’s value becomes a single “token” represented as a column vector.  This means each of Q, K, and V will have shape (28, 1).

The core attention calculation is performed: `scores = Q @ K.T / np.sqrt(Q.shape[1])`. This calculates the similarity between each pair of days in the window by taking the dot product of their corresponding vectors from `Q` and `K` (transposed). The result is scaled down by the square root of the dimension of the keys (`Q.shape[1]`) to prevent gradients from becoming too large during training.

The similarity scores are then passed through the `softmax` function to obtain attention weights: `weights = softmax(scores)`. These weights represent the relevance of each day in the window to every other day. Each row of the `weights` matrix represents a probability distribution over all days, indicating how much attention should be paid to each day when considering a specific day.

Finally, the attended representation is calculated: `attended = weights @ V`. This computes a weighted sum of the values (`V`), where the weights are determined by the attention mechanism. The result is a new representation of each day that incorporates information from all other days in the window, weighted by their relevance.

The code then prints the shape of the `weights` matrix and verifies that each row sums to approximately 1 (indicating a valid probability distribution). This demonstrates how self-attention allows the model to focus on different parts of the input sequence based on their importance for understanding the current day’s value.

In [ ]:
# first the raw material: the window itself
plt.figure(figsize=(CFG.img_dim1, CFG.img_dim2 * 0.6))
plt.plot(window, linewidth=2)
plt.title('the window: 28 days of item 1 (standardized)')
plt.xlabel('day in window')
plt.show()

# then what attention computed from it
plt.figure(figsize=(CFG.img_dim2 * 0.9, CFG.img_dim2 * 0.8))
plt.imshow(weights, cmap='viridis')
plt.colorbar(label='attention weight')
plt.title('attention weights: rows ask, columns answer')
plt.xlabel('key day j')
plt.ylabel('query day i')
plt.show()

This code visualizes the input window and the attention weights calculated from it, providing a clear understanding of how the self-attention mechanism operates on this specific example.

The first part creates a line plot of the `window` variable, which contains 28 days of standardized sales data for item 1. The plot is titled "the window: 28 days of item 1 (standardized)" and includes labels for the x-axis ("day in window") and y-axis (implicitly representing the standardized sales value). This visualization shows the raw input sequence that the attention mechanism processes.

The second part generates a heatmap visualization of the `weights` matrix, which represents the attention weights calculated by the self-attention mechanism. The plot is titled "attention weights: rows ask, columns answer" and uses the 'viridis' colormap to represent the weight values. A colorbar is added to indicate the mapping between colors and attention weights. The x-axis is labeled "key day j" and the y-axis is labeled "query day i," indicating that each row represents a query (asking which days are relevant) and each column represents a key (providing information about each day).

This heatmap allows for visual inspection of how much attention each day in the window pays to every other day. Brighter colors indicate higher attention weights, meaning those days are considered more relevant to each other. This visualization helps understand which parts of the input sequence the model focuses on when processing a particular day.

In [ ]:
def attend(x):
    """The four attention lines as a function: values in, mixed values out."""
    q = k = v = x.reshape(-1, 1)
    return (softmax(q @ k.T / np.sqrt(1)) @ v).ravel()

rng = np.random.default_rng(CFG.SEED)
perm = rng.permutation(len(window))       # scramble time
shuffled = window[perm]

print('attention output on the SHUFFLED window == shuffled output on the original?',
      np.allclose(attend(shuffled), attend(window)[perm]))

This code defines a function `attend` that encapsulates the self-attention mechanism and then tests its permutation invariance property.

The `attend` function takes an input array `x` and performs the following steps:

1.  It reshapes the input `x` into a column vector, creating matrices `q`, `k`, and `v` where each element of the original sequence becomes a single “token”.
2.  It calculates the attention weights by taking the dot product of `q` and the transpose of `k`, scaling it by the square root of 1 (which simplifies to no scaling in this case since the dimension is 1), and then applying the softmax function.
3.  It computes the attended representation by multiplying the attention weights with `v`.
4.  Finally, it flattens the resulting array using `.ravel()` and returns it.

Next, a random number generator (`rng`) is initialized with the seed specified in `CFG.SEED` to ensure reproducibility. A permutation of the indices of the `window` array is generated using `rng.permutation()`. This creates a shuffled version of the original time series data.

Finally, the code tests whether the attention mechanism is permutation invariant. It checks if applying the `attend` function to the shuffled window produces the same result as applying it to the original window and then permuting the output accordingly. The `np.allclose()` function compares the two results element-wise with a tolerance for floating-point errors, returning `True` if they are approximately equal and `False` otherwise.

The print statement displays whether the attention mechanism satisfies this property. Permutation invariance means that the order of elements in the input sequence should not affect the output of the attention mechanism, as long as the relative relationships between the elements remain the same. This is a desirable property for time series analysis, as it ensures that the model can learn patterns regardless of their specific position in the sequence.

In [ ]:
def sinusoidal_pe(length, depth):
    """The original Transformer position stamp: sines and cosines at many scales."""
    pos = np.arange(length)[:, None]          # position index down the rows
    i = np.arange(depth)[None, :]             # embedding dimension across columns
    angle = pos / np.power(10000.0, (2 * (i // 2)) / depth)
    return np.where(i % 2 == 0, np.sin(angle), np.cos(angle)).astype('float32')

pe = sinusoidal_pe(CFG.LOOK_BACK, CFG.D_MODEL)

plt.figure(figsize=(CFG.img_dim1, CFG.img_dim2 * 0.6))
plt.imshow(pe.T, aspect='auto', cmap='RdBu')
plt.colorbar(label='encoding value')
plt.xlabel('position (day in the 84-day window)')
plt.ylabel('embedding dimension')
plt.title('Sinusoidal positional encoding')
plt.show()

This code defines a function `sinusoidal_pe` that generates sinusoidal positional encodings, as originally used in the Transformer architecture, and then visualizes these encodings.

The `sinusoidal_pe` function takes two arguments: `length`, which specifies the length of the sequence (in this case, `CFG.LOOK_BACK`), and `depth`, which represents the embedding dimension (`CFG.D_MODEL`). It creates a matrix where each row corresponds to a position in the sequence and each column represents an embedding dimension. The values in the matrix are calculated using sine and cosine functions with different frequencies, allowing the model to distinguish between different positions in the input sequence.

Specifically:

1.  `pos = np.arange(length)[:, None]` creates a column vector of position indices from 0 to `length - 1`.
2.  `i = np.arange(depth)[None, :]` creates a row vector of embedding dimension indices from 0 to `depth - 1`.
3.  `angle = pos / np.power(10000.0, (2 * (i // 2)) / depth)` calculates the angles for the sine and cosine functions. The frequencies decrease as the embedding dimension increases, allowing the model to capture both fine-grained and coarse-grained positional information.
4.  `np.where(i % 2 == 0, np.sin(angle), np.cos(angle)).astype('float32')` uses a conditional statement to assign sine values to even embedding dimensions and cosine values to odd embedding dimensions. The result is cast to the `'float32'` data type.

The code then calls `sinusoidal_pe` with `CFG.LOOK_BACK` and `CFG.D_MODEL` to generate positional encodings for a window of 84 days with an embedding dimension of 64, storing the result in the `pe` variable.

Finally, it creates a heatmap visualization of the generated positional encodings using `plt.imshow()`. The plot is titled "Sinusoidal positional encoding" and includes labels for the x-axis ("position (day in the 84-day window)") and y-axis ("embedding dimension"). A colorbar is added to indicate the mapping between colors and encoding values. This visualization shows how different positions are encoded into different embedding dimensions, allowing the model to learn positional information.

# The transformer ladder

## 1. Vanilla transformer: timestamps as tokens

In [ ]:
def encoder_block(x, d_model, heads, ff_mult=2, dropout=0.1, name=None):
    """One transformer encoder block: self-attention + feed-forward,
    each wrapped in a residual connection and layer normalization."""
    att = MultiHeadAttention(num_heads=heads, key_dim=d_model // heads,
                             dropout=dropout,
                             name=None if name is None else f'{name}_mha')(x, x)
    x = LayerNormalization()(x + att)
    ff = Dense(d_model * ff_mult, activation='relu')(x)
    ff = Dense(d_model)(ff)
    return LayerNormalization()(x + ff)

This code defines a function `encoder_block` that implements a single transformer encoder block. This is a fundamental building block of the Transformer architecture used in many modern deep learning models, particularly for sequence processing tasks.

The function takes several arguments:

*   `x`: The input tensor to the encoder block.
*   `d_model`: The embedding dimension (the size of the hidden state).
*   `heads`: The number of attention heads in the multi-head attention mechanism.
*   `ff_mult`: A multiplier for the feed-forward network's hidden layer size (defaulting to 2).
*   `dropout`: The dropout rate used within the block (defaulting to 0.1).
*   `name`: An optional name for the encoder block, useful for debugging and visualization.

The function performs the following steps:

1.  **Multi-Head Attention:** It applies a multi-head attention mechanism (`MultiHeadAttention`) to the input `x`. The number of heads is specified by the `heads` argument, and the key dimension is calculated as `d_model // heads`. A dropout layer is included for regularization.
2.  **Residual Connection & Layer Normalization (1):** It adds the output of the multi-head attention mechanism to the original input `x` (residual connection) and then applies layer normalization (`LayerNormalization`) to stabilize training.
3.  **Feed-Forward Network:** It passes the result through a feed-forward network consisting of two dense layers with a ReLU activation function in between. The hidden layer size is determined by multiplying `d_model` by `ff_mult`.
4.  **Residual Connection & Layer Normalization (2):** It adds the output of the feed-forward network to the previous result (another residual connection) and then applies another layer normalization.

Finally, it returns the output of the encoder block, which is a transformed version of the input `x` that has been processed by both the self-attention mechanism and the feed-forward network. The use of residual connections and layer normalization helps to improve training stability and performance.

In [ ]:
def build_vanilla(look_back=CFG.LOOK_BACK, n_channels=CFG.N_ITEMS,
                  horizon=CFG.HORIZON, d_model=CFG.D_MODEL, head='pool'):
    """Vanilla time series transformer: one token per timestamp."""
    pe = sinusoidal_pe(look_back, d_model)
    with keras.device(DEVICE):
        inp = Input(shape=(look_back, n_channels))
        x = Dense(d_model, name='token_embed')(inp) + pe   # embed day, stamp position
        for i in range(CFG.N_BLOCKS):
            x = encoder_block(x, d_model, CFG.N_HEADS, name=f'attn{i + 1}')
        x = GlobalAveragePooling1D()(x) if head == 'pool' else Flatten()(x)
        out = Reshape((horizon, n_channels))(Dense(horizon * n_channels)(x))
        model = Model(inp, out)
    model.compile(loss='mse', optimizer=keras.optimizers.Adam(1e-3))
    return model

This code defines a function `build_vanilla` that constructs a basic time series transformer model for forecasting.

The function takes several arguments: `look_back`, the length of the input sequence; `n_channels`, the number of features or items being forecast; `horizon`, the length of the forecast horizon; `d_model`, the embedding dimension; and `head`, a string specifying the type of head to use for final prediction ('pool' for global average pooling, 'flatten' for flattening).

First, it generates sinusoidal positional encodings using the `sinusoidal_pe` function with the specified `look_back` and `d_model`. These encodings are added to the input embeddings to provide information about the position of each time step in the sequence.

Then, within a `keras.device(DEVICE)` context (ensuring model creation on the correct device), it defines the model architecture:

1.  **Input Layer:** An `Input` layer is created with shape (`look_back`, `n_channels`).
2.  **Token Embedding:** A dense layer (`Dense`) maps the input to a `d_model`-dimensional embedding space. The positional encodings are added to this embedding.
3.  **Encoder Blocks:** It stacks `CFG.N_BLOCKS` encoder blocks, each implemented by the `encoder_block` function. These blocks perform self-attention and feed-forward transformations on the input sequence.
4.  **Head Layer:** Based on the value of the `head` argument, either a global average pooling layer (`GlobalAveragePooling1D`) or a flattening layer (`Flatten`) is applied to reduce the output of the encoder blocks to a fixed-size vector.
5.  **Output Layer:** A dense layer maps the pooled/flattened representation to a vector of size `horizon * n_channels`. This vector is then reshaped into a tensor with shape (`horizon`, `n_channels`) using `Reshape` to represent the forecasted time series for each item over the forecast horizon.
6.  **Model Creation:** A Keras `Model` is created, specifying the input and output layers.

Finally, the model is compiled using the mean squared error (`mse`) loss function and the Adam optimizer with a learning rate of 1e-3. The function returns the constructed Keras model. This creates a standard transformer architecture suitable for time series forecasting tasks.

In [ ]:
keras.utils.set_random_seed(CFG.SEED)
vanilla_model = fit_timed('vanilla transformer', build_vanilla(),
                          X_train, y_train, X_val, y_val)

results['vanilla transformer'] = score_test(vanilla_model.predict(X_test, verbose=0))
results['vanilla transformer']

This code trains a vanilla time series transformer model and evaluates its performance on the test set.

First, it sets the random seed using `keras.utils.set_random_seed(CFG.SEED)` to ensure reproducibility of results.

Then, it calls the `build_vanilla` function to create a vanilla transformer model with default parameters. The resulting model is trained using the `fit_timed` function, passing in the training data (`X_train`, `y_train`), validation data (`X_val`, `y_val`), and assigning it the name 'vanilla transformer' for tracking purposes. The trained model is stored in the variable `vanilla_model`.

Next, it uses the trained `vanilla_model` to generate predictions on the test data (`X_test`) using `vanilla_model.predict()`, with output suppressed by setting `verbose=0`.

The performance of the vanilla transformer model is then evaluated using the `score_test` function, which calculates MAE and RMSE on the test set. The resulting metrics are stored in the `results` dictionary under the key 'vanilla transformer'.

Finally, it prints the contents of `results['vanilla transformer']`, displaying the calculated MAE and RMSE for the vanilla transformer model. This provides a quantitative assessment of its forecasting accuracy.

In [ ]:
keras.utils.set_random_seed(CFG.SEED)
vanilla_flat = fit_timed('vanilla transformer (flat head)',
                         build_vanilla(head='flat'),
                         X_train, y_train, X_val, y_val)

results['vanilla transformer (flat head)'] = score_test(
    vanilla_flat.predict(X_test, verbose=0))
results['vanilla transformer (flat head)']

This code trains a vanilla time series transformer model with a flattened output head and evaluates its performance on the test set. This is similar to the previous step but explores a different configuration for the final prediction layer.

First, it sets the random seed using `keras.utils.set_random_seed(CFG.SEED)` to ensure reproducibility of results.

Then, it calls the `build_vanilla` function with the `head` argument set to 'flat'. This creates a vanilla transformer model that uses a flattening layer instead of global average pooling for the final prediction head. The resulting model is trained using the `fit_timed` function, passing in the training data (`X_train`, `y_train`), validation data (`X_val`, `y_val`), and assigning it the name 'vanilla transformer (flat head)' for tracking purposes. The trained model is stored in the variable `vanilla_flat`.

Next, it uses the trained `vanilla_flat` model to generate predictions on the test data (`X_test`) using `vanilla_flat.predict()`, with output suppressed by setting `verbose=0`.

The performance of the vanilla transformer model with a flat head is then evaluated using the `score_test` function, which calculates MAE and RMSE on the test set. The resulting metrics are stored in the `results` dictionary under the key 'vanilla transformer (flat head)'.

Finally, it prints the contents of `results['vanilla transformer (flat head)']`, displaying the calculated MAE and RMSE for this model configuration. This allows for comparison between the performance of the vanilla transformer with a pooling head versus a flattened head.

In [ ]:
# what did temporal attention learn? pull the map out of the trained model
tokens = Model(vanilla_model.input,
               vanilla_model.get_layer('token_embed').output).predict(
                   X_test[:1], verbose=0) + sinusoidal_pe(CFG.LOOK_BACK, CFG.D_MODEL)

_, att_scores = vanilla_model.get_layer('attn1_mha')(
    tokens, tokens, return_attention_scores=True)
att_scores = keras.ops.convert_to_numpy(att_scores)   # (1, heads, 84, 84)

plt.figure(figsize=(CFG.img_dim2 * 0.9, CFG.img_dim2 * 0.8))
plt.imshow(att_scores[0].mean(axis=0), cmap='viridis')
plt.colorbar(label='attention weight (head average)')
plt.xlabel('key day (0 = twelve weeks ago, 83 = yesterday)')
plt.ylabel('query day')
plt.title('Vanilla transformer, block-1 attention over the 84-day window')
plt.show()

This code extracts and visualizes the attention weights learned by the first attention layer in the trained vanilla transformer model. This provides insights into which parts of the input sequence the model focuses on when making predictions.

First, it creates a new Keras `Model` named `tokens`. This model takes the input of the original `vanilla_model` and outputs the output of the 'token_embed' layer (the embedding layer). It then uses this model to predict the embeddings for the first sample in the test data (`X_test[:1]`). The sinusoidal positional encodings are added to these embeddings.

Next, it retrieves the attention scores from the first multi-head attention layer ('attn1\_mha') of the `vanilla_model`. It calls this layer with the embedded tokens as both query and key inputs, requesting the return of attention scores using `return_attention_scores=True`. The resulting `att_scores` tensor has shape (1, heads, 84, 84), where 1 represents the batch size, `heads` is the number of attention heads, and 84x84 represents the attention weights between each pair of days in the 84-day window.

The attention scores are then converted to a NumPy array using `keras.ops.convert_to_numpy()`.

Finally, it creates a heatmap visualization of the average attention weights across all heads. It calculates the mean attention weight for each day combination by averaging over the heads (`att_scores[0].mean(axis=0)`). The resulting 84x84 matrix is displayed as a heatmap using `plt.imshow()`. The plot is titled "Vanilla transformer, block-1 attention over the 84-day window" and includes labels for the x-axis ("key day") and y-axis ("query day"). A colorbar indicates the mapping between colors and attention weights. This visualization shows which days in the input sequence are most relevant to each other according to the model's learned attention mechanism.

## 2. the linear map

In [ ]:
def build_linear(look_back=CFG.LOOK_BACK, n_channels=CFG.N_ITEMS,
                 horizon=CFG.HORIZON):
    """One shared linear map from each channel's history to its future."""
    with keras.device(DEVICE):
        inp = Input(shape=(look_back, n_channels))
        x = Permute((2, 1))(inp)          # (channels, time): each channel a row
        x = Dense(horizon)(x)             # the same 84 -> 14 map for every channel
        out = Permute((2, 1))(x)          # back to (time, channels)
        model = Model(inp, out)
    model.compile(loss='mse', optimizer=keras.optimizers.Adam(1e-3))
    return model

This code defines a function `build_linear` that constructs a simple linear forecasting model. This serves as a baseline to compare against more complex models like the transformer.

The function takes arguments for `look_back`, `n_channels`, and `horizon`, defining the input sequence length, number of features, and forecast horizon respectively.

Inside a `keras.device(DEVICE)` context (ensuring model creation on the correct device), it defines the model architecture:

1.  **Input Layer:** An `Input` layer is created with shape (`look_back`, `n_channels`).
2.  **Permute Layer:** A `Permute` layer rearranges the dimensions of the input tensor from (time, channels) to (channels, time). This effectively treats each channel (item) as a separate row in the input data.
3.  **Dense Layer:** A dense (fully connected) layer maps each channel’s history (`look_back`) to the forecast horizon (`horizon`). Importantly, this is *the same* linear transformation applied to every channel independently. This represents a simple linear model for each item.
4.  **Permute Layer:** Another `Permute` layer rearranges the dimensions back from (channels, time) to (time, channels), restoring the original data format.

Finally, a Keras `Model` is created with the input and output layers. The model is compiled using the mean squared error (`mse`) loss function and the Adam optimizer with a learning rate of 1e-3. The function returns this constructed linear model. This model essentially learns a separate linear regression for each item in the dataset, mapping its historical values to future predictions.

In [ ]:
keras.utils.set_random_seed(CFG.SEED)
linear_model = fit_timed('linear (channel-independent)', build_linear(),
                         X_train, y_train, X_val, y_val)

results['linear (channel-independent)'] = score_test(
    linear_model.predict(X_test, verbose=0))
results['linear (channel-independent)']

This code trains the linear forecasting model defined in the previous step and evaluates its performance on the test set.

First, it sets the random seed using `keras.utils.set_random_seed(CFG.SEED)` to ensure reproducibility of results.

Then, it calls the `build_linear` function to create a linear model with default parameters. The resulting model is trained using the `fit_timed` function, passing in the training data (`X_train`, `y_train`), validation data (`X_val`, `y_val`), and assigning it the name 'linear (channel-independent)' for tracking purposes. The trained model is stored in the variable `linear_model`.

Next, it uses the trained `linear_model` to generate predictions on the test data (`X_test`) using `linear_model.predict()`, with output suppressed by setting `verbose=0`.

The performance of the linear model is then evaluated using the `score_test` function, which calculates MAE and RMSE on the test set. The resulting metrics are stored in the `results` dictionary under the key 'linear (channel-independent)'.

Finally, it prints the contents of `results['linear (channel-independent)']`, displaying the calculated MAE and RMSE for this model configuration. This provides a quantitative assessment of the linear model’s forecasting accuracy, which will be used as a baseline for comparison with more complex models like the transformer.

## 3. PatchTST

In [ ]:
def fold_channels(X, y):
    """(N, L, C) -> (N*C, L, 1): every channel becomes its own univariate sample."""
    n, look_back, n_ch = X.shape
    X_flat = X.transpose(0, 2, 1).reshape(n * n_ch, look_back, 1)
    y_flat = y.transpose(0, 2, 1).reshape(n * n_ch, y.shape[1], 1)
    return X_flat, y_flat

def unfold_channels(pred, n_windows):
    """(N*C, H, 1) -> (N, H, C): reassemble per-channel forecasts into a panel."""
    horizon = pred.shape[1]
    return pred.reshape(n_windows, -1, horizon).transpose(0, 2, 1)

X_train_ci, y_train_ci = fold_channels(X_train, y_train)
X_val_ci, y_val_ci = fold_channels(X_val, y_val)
X_test_ci, _ = fold_channels(X_test, y_test)   # test labels stay in panel shape

print(f'channel-independent training set: {X_train_ci.shape} '
      f'(was {X_train.shape})')

This code defines two functions, `fold_channels` and `unfold_channels`, to reshape the data for channel-independent modeling – meaning each item’s time series is treated as a separate univariate forecasting problem. It then applies these functions to the training and validation datasets.

The `fold_channels` function takes input features (`X`) and target values (`y`) with shape (N, L, C), where N is the number of samples, L is the lookback period, and C is the number of channels (items). It reshapes these tensors into a format suitable for training individual models on each channel. Specifically:

1.  It transposes `X` from (N, L, C) to (N, C, L).
2.  It then reshapes `X` to (N * C, L, 1), effectively flattening the channels into a single dimension and creating univariate time series for each channel.
3.  The same process is applied to `y`, reshaping it from (N, H, C) to (N * C, H, 1), where H is the horizon length.

The `unfold_channels` function performs the reverse operation. It takes predictions (`pred`) with shape (N * C, H, 1) and reshapes them back into a panel format of shape (N, H, C). This allows for reassembling the per-channel forecasts into a single multi-channel time series prediction.

The code then applies `fold_channels` to the training (`X_train`, `y_train`) and validation (`X_val`, `y_val`) datasets, creating `X_train_ci`, `y_train_ci`, `X_val_ci`, and `y_val_ci`. It also applies it to the test features (`X_test`), but keeps the test labels in their original panel shape.

Finally, it prints the shape of the reshaped training set (`X_train_ci`) along with its original shape for comparison, confirming that the reshaping operation has been performed correctly. This prepares the data for a channel-independent modeling approach where each item is treated as an independent time series forecasting problem.

In [ ]:
# what patching looks like: one window cut into six two-week tokens
demo = z[test_starts[0] - CFG.LOOK_BACK:test_starts[0], 0]
n_patches = CFG.LOOK_BACK // CFG.PATCH_LEN

plt.figure(figsize=(CFG.img_dim1, CFG.img_dim2 * 0.6))
for p in range(n_patches):
    seg = range(p * CFG.PATCH_LEN, (p + 1) * CFG.PATCH_LEN)
    plt.plot(seg, demo[seg], marker='o', markersize=3.5,
             label=f'patch {p + 1} = token {p + 1}')
    plt.axvline(p * CFG.PATCH_LEN, color='gray', linewidth=0.6, linestyle=':')
plt.title('Patching: 84 days -> 6 tokens of 14 days each (item 1, one test window)')
plt.xlabel('day in window')
plt.legend(ncol=3)
plt.show()

This code visualizes the process of patching a time series into fixed-length segments, which are then treated as tokens for a transformer model.

It begins by extracting a segment of the standardized data (`z`) corresponding to one test window (84 days) for item 1. This segment is stored in the `demo` variable.

The number of patches (tokens) is calculated by dividing the lookback period (`CFG.LOOK_BACK`) by the patch length (`CFG.PATCH_LEN`). In this case, 84 / 14 = 6 patches.

A plot is created to illustrate how the time series is divided into these patches. The code iterates through each patch:

1.  For each patch `p`, it defines a segment of indices (`seg`) representing the days belonging to that patch.
2.  It plots the values from the `demo` array corresponding to those indices, using markers and labels indicating the patch number.
3.  Vertical dashed lines are added at the boundaries between patches to visually separate them.

The plot is titled "Patching: 84 days -> 6 tokens of 14 days each (item 1, one test window)" and includes labels for the x-axis ("day in window") and a legend with patch numbers arranged in three columns. This visualization demonstrates how the continuous time series data is divided into discrete segments (patches) that can be treated as individual tokens by a transformer model. Each token represents a fixed duration of time (14 days in this case).

In [ ]:
def build_patchtst(look_back=CFG.LOOK_BACK, horizon=CFG.HORIZON,
                   patch_len=CFG.PATCH_LEN, d_model=CFG.D_MODEL):
    """PatchTST-lite: patch tokens, learned positions, channel-independent."""
    n_patches = look_back // patch_len
    with keras.device(DEVICE):
        inp = Input(shape=(look_back, 1))
        x = Reshape((n_patches, patch_len))(inp)     # cut history into patches
        x = Dense(d_model)(x)                        # embed each patch (a shape!)
        positions = keras.Variable(                  # learned positional encoding
            np.random.normal(0, 0.02, (n_patches, d_model)).astype('float32'),
            name='patch_positions')
        x = x + positions
        for i in range(CFG.N_BLOCKS):
            x = encoder_block(x, d_model, CFG.N_HEADS, name=f'attn{i + 1}')
        x = Flatten()(x)                             # all patch tokens feed the head
        out = Reshape((horizon, 1))(Dense(horizon)(x))
        model = Model(inp, out)
    model.compile(loss='mse', optimizer=keras.optimizers.Adam(1e-3))
    return model

This code defines a function `build_patchtst` that constructs a PatchTST-lite model for time series forecasting. This architecture divides the input sequence into patches, embeds each patch, and then uses transformer encoder blocks to learn relationships between these patches.

The function takes arguments for `look_back`, `horizon`, `patch_len`, and `d_model`.

Inside a `keras.device(DEVICE)` context:

1.  **Input Layer:** An `Input` layer is created with shape (`look_back`, 1). Note that the input is assumed to be univariate (single channel) here, which differs from previous models.
2.  **Patching:** The input sequence is divided into patches using a `Reshape` layer. The number of patches is calculated as `look_back // patch_len`. This transforms the input from (L, 1) to (N, P), where N is the number of patches and P is the patch length.
3.  **Patch Embedding:** A dense layer maps each patch into a `d_model`-dimensional embedding space. This effectively represents each patch as a vector.
4.  **Learned Positional Encoding:** Learned positional embeddings are added to the patch embeddings. A Keras `Variable` is created and initialized with random values, representing the learned positions for each patch. These positions are added to the patch embeddings to provide information about their order in the sequence.
5.  **Encoder Blocks:** The embedded patches are passed through a series of `CFG.N_BLOCKS` encoder blocks (implemented by the `encoder_block` function). Each block performs self-attention and feed-forward transformations on the patch embeddings.
6.  **Flattening & Output Layer:** The output of the encoder blocks is flattened using a `Flatten` layer, combining all patch tokens into a single vector. This vector is then passed through a dense layer to predict the future values for the specified horizon. Finally, another `Reshape` layer transforms the output into the desired shape (`horizon`, 1).
7.  **Model Creation & Compilation:** A Keras `Model` is created with the input and output layers. The model is compiled using the mean squared error (`mse`) loss function and the Adam optimizer with a learning rate of 1e-3.

The function returns the constructed PatchTST-lite model. This architecture leverages patching to reduce sequence length, learned positional embeddings to capture temporal information, and transformer encoder blocks to learn complex relationships between patches for improved forecasting performance. The channel independence is enforced by assuming a single input channel.

In [ ]:
keras.utils.set_random_seed(CFG.SEED)
patch_model = fit_timed('PatchTST-lite', build_patchtst(),
                        X_train_ci, y_train_ci, X_val_ci, y_val_ci, batch=256)

patch_pred = unfold_channels(patch_model.predict(X_test_ci, verbose=0),
                             len(test_starts))
results['PatchTST-lite'] = score_test(patch_pred)
results['PatchTST-lite']

This code trains a PatchTST-lite model and evaluates its performance on the test set.

First, it sets the random seed using `keras.utils.set_random_seed(CFG.SEED)` to ensure reproducibility of results.

Then, it calls the `build_patchtst` function to create a PatchTST-lite model with default parameters. The resulting model is trained using the `fit_timed` function, passing in the channel-independent training data (`X_train_ci`, `y_train_ci`), validation data (`X_val_ci`, `y_val_ci`), and setting the batch size to 256. The trained model is stored in the variable `patch_model`.

Next, it uses the trained `patch_model` to generate predictions on the channel-independent test data (`X_test_ci`) using `patch_model.predict()`, with output suppressed by setting `verbose=0`.  The predicted values are then reshaped back into a panel format using the `unfold_channels` function, taking the number of test starts as input to determine the correct dimensions. The unfolded predictions are stored in the `patch_pred` variable.

The performance of the PatchTST-lite model is then evaluated using the `score_test` function, which calculates MAE and RMSE on the test set. The resulting metrics are stored in the `results` dictionary under the key 'PatchTST-lite'.

Finally, it prints the contents of `results['PatchTST-lite']`, displaying the calculated MAE and RMSE for this model configuration. This provides a quantitative assessment of the PatchTST-lite model’s forecasting accuracy.

## 4. iTransformer

In [ ]:
def build_itransformer(look_back=CFG.LOOK_BACK, n_channels=CFG.N_ITEMS,
                       horizon=CFG.HORIZON, d_model=2 * CFG.D_MODEL):
    """iTransformer-lite: one token per CHANNEL, attention across channels."""
    with keras.device(DEVICE):
        inp = Input(shape=(look_back, n_channels))
        x = Permute((2, 1))(inp)                       # the inversion: (channels, time)
        x = Dense(d_model, name='variate_embed')(x)    # a channel's WHOLE history -> one token
        for i in range(CFG.N_BLOCKS):
            x = encoder_block(x, d_model, CFG.N_HEADS, name=f'attn{i + 1}')
        x = Dense(horizon)(x)                          # per-channel forecast head
        out = Permute((2, 1))(x)                       # back to (time, channels)
        model = Model(inp, out)
    model.compile(loss='mse', optimizer=keras.optimizers.Adam(1e-3))
    return model

This code defines a function `build_itransformer` that constructs an iTransformer-lite model for multivariate time series forecasting. This architecture treats each channel (item) as a separate token and uses attention to learn relationships *between* these channels.

The function takes arguments for `look_back`, `n_channels`, `horizon`, and `d_model`. The embedding dimension `d_model` is doubled compared to previous models (`2 * CFG.D_MODEL`).

Inside a `keras.device(DEVICE)` context:

1.  **Input Layer:** An `Input` layer is created with shape (`look_back`, `n_channels`).
2.  **Permutation:** A `Permute` layer rearranges the dimensions of the input tensor from (time, channels) to (channels, time). This is crucial because it prepares the data for treating each channel as a separate sequence.
3.  **Variate Embedding:** A dense layer maps each channel’s entire history (`look_back`) into a `d_model`-dimensional embedding space. This creates a single token representing the historical information for each item (channel).
4.  **Encoder Blocks:** The embedded channels are passed through a series of `CFG.N_BLOCKS` encoder blocks (implemented by the `encoder_block` function). These blocks perform self-attention, allowing the model to learn relationships between different channels. Because the input is now (channels, time), the attention mechanism operates across channels.
5.  **Per-Channel Forecast Head:** A dense layer maps the output of the encoder blocks to a vector of size `horizon`, representing the forecast for each channel. This creates per-channel forecasts directly.
6.  **Permutation:** Another `Permute` layer rearranges the dimensions back from (channels, time) to (time, channels), restoring the original data format.
7.  **Model Creation & Compilation:** A Keras `Model` is created with the input and output layers. The model is compiled using the mean squared error (`mse`) loss function and the Adam optimizer with a learning rate of 1e-3.

The function returns the constructed iTransformer-lite model. This architecture differs from previous models by treating each channel as an independent sequence and using attention to learn relationships between them, potentially capturing cross-channel dependencies that could improve forecasting accuracy.

In [ ]:
keras.utils.set_random_seed(CFG.SEED)
itransformer_model = fit_timed('iTransformer-lite', build_itransformer(),
                               X_train, y_train, X_val, y_val)

results['iTransformer-lite'] = score_test(
    itransformer_model.predict(X_test, verbose=0))
results['iTransformer-lite']

This code trains an iTransformer-lite model and evaluates its performance on the test set.

First, it sets the random seed using `keras.utils.set_random_seed(CFG.SEED)` to ensure reproducibility of results.

Then, it calls the `build_itransformer` function to create an iTransformer-lite model with default parameters. The resulting model is trained using the `fit_timed` function, passing in the training data (`X_train`, `y_train`), validation data (`X_val`, `y_val`). The trained model is stored in the variable `itransformer_model`.

Next, it uses the trained `itransformer_model` to generate predictions on the test data (`X_test`) using `itransformer_model.predict()`, with output suppressed by setting `verbose=0`.

The performance of the iTransformer-lite model is then evaluated using the `score_test` function, which calculates MAE and RMSE on the test set. The resulting metrics are stored in the `results` dictionary under the key 'iTransformer-lite'.

Finally, it prints the contents of `results['iTransformer-lite']`, displaying the calculated MAE and RMSE for this model configuration. This provides a quantitative assessment of the iTransformer-lite model’s forecasting accuracy, allowing comparison with other models tested in this notebook.

In [ ]:
    # the learned channel-to-channel attention, next to the raw correlation matrix
    variate_tokens = Model(itransformer_model.input,
                        itransformer_model.get_layer('variate_embed').output
                        ).predict(X_test[:1], verbose=0)
    _, channel_att = itransformer_model.get_layer('attn1_mha')(
        variate_tokens, variate_tokens, return_attention_scores=True)
    channel_att = keras.ops.convert_to_numpy(channel_att)

    # the co-movement the data offers ...
    plt.figure(figsize=(CFG.img_dim2 * 0.9, CFG.img_dim2 * 0.8))
    plt.imshow(corr, cmap='viridis', vmin=0, vmax=1)
    plt.colorbar(label='Pearson correlation')
    plt.title('Pearson correlation between items')
    plt.xlabel('item')
    plt.ylabel('item')
    plt.show()

    # ... and the relationships the model chose to learn
    plt.figure(figsize=(CFG.img_dim2 * 0.9, CFG.img_dim2 * 0.8))
    plt.imshow(channel_att[0].mean(axis=0), cmap='viridis')
    plt.colorbar(label='attention weight (head average)')
    plt.title('iTransformer block-1 attention between items')
    plt.xlabel('item')
    plt.ylabel('item')
    plt.show()

This code visualizes the channel-to-channel attention learned by the first attention layer in the trained iTransformer model, comparing it to the raw correlation matrix of the input data.

First, it extracts the variate embeddings (the output of the 'variate_embed' layer) for a single sample from the test set (`X_test[:1]`) using a Keras `Model` created specifically for this purpose. These embeddings represent each channel’s historical information as a vector.

Next, it retrieves the attention scores from the first multi-head attention layer ('attn1\_mha') of the `itransformer_model`. It calls this layer with the variate tokens as both query and key inputs, requesting the return of attention scores using `return_attention_scores=True`. The resulting `channel_att` tensor has shape (1, heads, C, C), where 1 represents the batch size, `heads` is the number of attention heads, and C x C represents the attention weights between each pair of channels.

The attention scores are then converted to a NumPy array using `keras.ops.convert_to_numpy()`.

Then, it displays the raw correlation matrix calculated earlier in the notebook using `plt.imshow()`. This provides a baseline for understanding the inherent relationships between items in the dataset. The plot is titled "Pearson correlation between items" and includes appropriate labels.

Finally, it creates another heatmap visualization of the average attention weights across all heads from the iTransformer model. It calculates the mean attention weight for each pair of channels by averaging over the heads (`channel_att[0].mean(axis=0)`). The resulting C x C matrix is displayed as a heatmap using `plt.imshow()`. The plot is titled "iTransformer block-1 attention between items" and includes labels for the x and y axes (representing items). This visualization shows which channels the model considers most relevant to each other, based on its learned attention mechanism.

By comparing these two visualizations, one can assess whether the iTransformer model learns relationships that align with the inherent correlations in the data or discovers new dependencies not captured by simple correlation analysis.

# When transformers win: the lookback sweep

In [ ]:
SWEEP_LOOKBACKS = [28, 84, 182, 364]     # 4, 12, 26 and 52 weeks of history

sweep_rmse = {'GRU': [], 'iTransformer': []}
sweep_secs = {'GRU': [], 'iTransformer': []}

for lb in SWEEP_LOOKBACKS:
    s_tr, s_va, s_te = split_starts(N_DAYS, lb, CFG.HORIZON, val_start, test_start)
    Xs_tr, ys_tr = make_windows(z, s_tr, lb, CFG.HORIZON)
    Xs_va, ys_va = make_windows(z, s_va, lb, CFG.HORIZON)
    Xs_te, _ = make_windows(z, s_te, lb, CFG.HORIZON)

    for label, build in [('GRU', lambda: build_rnn(GRU, look_back=lb)),
                         ('iTransformer', lambda: build_itransformer(look_back=lb))]:
        keras.utils.set_random_seed(CFG.SEED)
        model = fit_timed(f'{label} (L={lb})', build(), Xs_tr, ys_tr, Xs_va, ys_va)
        pred = model.predict(Xs_te, verbose=0)
        sweep_rmse[label].append(score_in_units(
            pred, s_te, CFG.HORIZON, panel.values, train_mean, train_std)['RMSE'])
        sweep_secs[label].append(train_times[f'{label} (L={lb})'])

pd.DataFrame(sweep_rmse, index=pd.Index(SWEEP_LOOKBACKS, name='lookback days'))

This code performs a hyperparameter sweep to evaluate the impact of different lookback periods on the performance and training time of both GRU and iTransformer models.

It begins by defining a list `SWEEP_LOOKBACKS` containing four different lookback values: 28, 84, 182, and 364 days (representing 4, 12, 26, and 52 weeks of history).

Two dictionaries, `sweep_rmse` and `sweep_secs`, are initialized to store the Root Mean Squared Error (RMSE) and training time for each model and lookback period.

The code then iterates through each lookback value in `SWEEP_LOOKBACKS`:

1.  **Data Splitting & Windowing:** It calls `split_starts` to generate train, validation, and test starting indices based on the current lookback period. Then, it uses `make_windows` to create the corresponding input-output pairs (X, y) for each set.
2.  **Model Training & Evaluation:** It iterates through two models: GRU and iTransformer. For each model:
    *   It sets the random seed using `keras.utils.set_random_seed(CFG.SEED)` to ensure reproducibility.
    *   It calls `fit_timed` to train the model with the specified lookback period, passing in the training data, validation data, and a descriptive name.
    *   It generates predictions on the test set using `model.predict()`.
    *   It calculates the RMSE of the predictions using `score_in_units` and appends it to the `sweep_rmse` dictionary for the corresponding model.
    *   It retrieves the training time from the `train_times` dictionary (populated by `fit_timed`) and appends it to the `sweep_secs` dictionary for the corresponding model.

Finally, it creates a Pandas DataFrame using `pd.DataFrame()` to organize the results. The DataFrame has the lookback days as its index (`SWEEP_LOOKBACKS`) and columns representing the RMSE values for each model (GRU and iTransformer). This DataFrame provides a clear overview of how performance changes with different lookback periods, allowing for comparison between the two models.

In [ ]:
# accuracy as a function of visible history
plt.figure(figsize=(CFG.img_dim1 * 0.7, CFG.img_dim2 * 0.8))
for label, marker in [('GRU', 'o'), ('iTransformer', 's')]:
    plt.plot(SWEEP_LOOKBACKS, sweep_rmse[label], marker=marker, linewidth=2,
             label=label)
plt.title('accuracy: test RMSE vs lookback window')
plt.ylabel('RMSE (units sold)')
plt.xlabel('lookback window (days)')
plt.xticks(SWEEP_LOOKBACKS)
plt.legend()
plt.show()

# training cost as a function of visible history
plt.figure(figsize=(CFG.img_dim1 * 0.7, CFG.img_dim2 * 0.8))
for label, marker in [('GRU', 'o'), ('iTransformer', 's')]:
    plt.plot(SWEEP_LOOKBACKS, sweep_secs[label], marker=marker, linewidth=2,
             label=label)
plt.title('cost: training wall-clock vs lookback window')
plt.ylabel('seconds')
plt.xlabel('lookback window (days)')
plt.xticks(SWEEP_LOOKBACKS)
plt.legend()
plt.show()

This code generates two plots visualizing the results of the hyperparameter sweep, showing how accuracy and training cost vary with different lookback periods for both GRU and iTransformer models.

The first plot displays the relationship between test RMSE (Root Mean Squared Error) and the lookback window size. It iterates through each model (GRU and iTransformer), plotting the corresponding RMSE values from `sweep_rmse` against the lookback days in `SWEEP_LOOKBACKS`. Different markers are used for each model to distinguish them on the plot. The plot is titled "accuracy: test RMSE vs lookback window," with appropriate labels for the x-axis (lookback window in days) and y-axis (RMSE in units sold).  The x-ticks are set to match the values in `SWEEP_LOOKBACKS`. A legend is included to identify each model.

The second plot visualizes the relationship between training time (in seconds) and the lookback window size. It follows a similar structure as the first plot, but uses the training times from `sweep_secs` instead of RMSE values. The plot is titled "cost: training wall-clock vs lookback window," with appropriate labels for the x-axis (lookback window in days) and y-axis (seconds).  The x-ticks are set to match the values in `SWEEP_LOOKBACKS`. A legend is included to identify each model.

These plots allow for a direct comparison of how increasing the amount of historical data used as input affects both the accuracy and computational cost of the GRU and iTransformer models, helping to determine an optimal lookback period for this specific forecasting task.

# When transformers are overkill


In [ ]:
# task constants for the rematch
LOOK_BACK_B = 28      # four weeks of history
HORIZON_B = 7         # forecast one week ahead
TEST_DAYS_B = 365     # final year held out
VAL_DAYS_B = 365      # year before that for early stopping
RNN_UNITS_B = 32      # smaller data, smaller recurrent state

temps = pd.read_csv(CFG.data_folder + 'daily-min-temperatures.csv',
                    parse_dates=['Date'], index_col='Date')['Temp'].astype('float32')
N_DAYS_B = len(temps)

test_start_b = N_DAYS_B - TEST_DAYS_B
val_start_b = test_start_b - VAL_DAYS_B
mean_b = temps.values[:val_start_b].mean()
std_b = temps.values[:val_start_b].std()
z_temps = ((temps.values - mean_b) / std_b)[:, None]   # (days, 1): one channel

tr_b, va_b, te_b = split_starts(N_DAYS_B, LOOK_BACK_B, HORIZON_B,
                                val_start_b, test_start_b, test_stride=1)
Xb_train, yb_train = make_windows(z_temps, tr_b, LOOK_BACK_B, HORIZON_B)
Xb_val, yb_val = make_windows(z_temps, va_b, LOOK_BACK_B, HORIZON_B)
Xb_test, _ = make_windows(z_temps, te_b, LOOK_BACK_B, HORIZON_B)   # scored on raw

print(f'{N_DAYS_B} days of temperatures, '
      f'windows: {len(tr_b)} train / {len(va_b)} val / {len(te_b)} test')

This code sets up a new forecasting task using daily minimum temperature data as the target variable. It defines constants specific to this task and prepares the data for model training and evaluation.

It begins by defining several task-specific constants: `LOOK_BACK_B` (28 days of history), `HORIZON_B` (forecast one week ahead), `TEST_DAYS_B` (365 days for testing, representing a full year), `VAL_DAYS_B` (365 days for validation, also a full year), and `RNN_UNITS_B` (reduced to 32 due to the simpler dataset).

Next, it loads daily minimum temperature data from a CSV file named 'daily-min-temperatures.csv' using `pd.read_csv()`. The 'Date' column is parsed as datetime objects and set as the index of the DataFrame. Only the 'Temp' column is selected and converted to the `'float32'` data type.

The total number of days in the temperature dataset (`N_DAYS_B`) is calculated using `len(temps)`.

Then, it calculates the starting indices for the test and validation sets based on `TEST_DAYS_B` and `VAL_DAYS_B`. The mean and standard deviation are computed from the training data (up to `val_start_b`).  The temperature data is standardized by subtracting the mean and dividing by the standard deviation, resulting in a single-channel time series (`z_temps`) with shape (days, 1).

It calls `split_starts` to generate train, validation, and test starting indices based on the new task’s constants. Then, it uses `make_windows` to create input-output pairs for each set. The test labels are not needed in windowed format so they are discarded.

Finally, it prints information about the dataset size and the number of samples in each split (train, validation, and test). This prepares the temperature data for training and evaluating forecasting models with a different configuration than the previous item-level sales task.

In [ ]:
def score_test_b(pred_z):
    """Score standardized predictions on the temperature test year, in deg C."""
    return score_in_units(pred_z, te_b, HORIZON_B, temps.values, mean_b, std_b)

results_b = {}

persistence = np.repeat(z_temps[te_b - 1][:, None, :], HORIZON_B, axis=1)
results_b['persistence'] = score_test_b(persistence)

naive_365 = np.stack([z_temps[s - 365:s + HORIZON_B - 365] for s in te_b])
results_b['seasonal naive (lag 365)'] = score_test_b(naive_365)

results_b

This code defines a scoring function specific to the temperature forecasting task and establishes baseline models for comparison.

The `score_test_b` function is defined to evaluate predictions on the temperature dataset. It takes standardized predictions (`pred_z`) as input and uses the `score_in_units` function with parameters tailored to this task: the test starting indices (`te_b`), the forecast horizon (`HORIZON_B`), the original temperature values (`temps.values`), the mean (`mean_b`), and the standard deviation (`std_b`). This ensures that the evaluation metrics (MAE and RMSE) are calculated in degrees Celsius, reflecting the scale of the target variable.

A dictionary `results_b` is initialized to store the performance metrics of different models for this task.

Two baseline forecasting models are created:

1.  **Persistence:** This model predicts future values by simply repeating the last observed value for each time step. It uses `np.repeat()` to replicate the previous day’s temperature (`z_temps[te_b - 1]`) across the forecast horizon (`HORIZON_B`).
2.  **Seasonal Naive (Lag 365):** This model predicts future values by repeating the temperature from 365 days ago (approximately one year). It extracts a slice of data representing the previous year’s temperatures for each test origin and stacks them to create the predictions.

The performance of each baseline model is evaluated using `score_test_b`, and the resulting metrics are stored in the `results_b` dictionary, keyed by the model name.

Finally, the code displays the `results_b` dictionary, which contains the evaluation metrics for the persistence and seasonal naive baselines. These baselines provide a simple point of comparison to assess whether more sophisticated models offer significant improvements in forecasting accuracy for temperature data.

In [ ]:
contenders = {
    'GRU': lambda: build_rnn(GRU, LOOK_BACK_B, 1, HORIZON_B, units=RNN_UNITS_B),
    'LSTM': lambda: build_rnn(LSTM, LOOK_BACK_B, 1, HORIZON_B, units=RNN_UNITS_B),
    'vanilla transformer': lambda: build_vanilla(LOOK_BACK_B, 1, HORIZON_B,
                                                 head='flat'),
    'PatchTST-lite': lambda: build_patchtst(LOOK_BACK_B, HORIZON_B, patch_len=7),
    'linear': lambda: build_linear(LOOK_BACK_B, 1, HORIZON_B),
}

for name, build in contenders.items():
    keras.utils.set_random_seed(CFG.SEED)
    model = fit_timed(f'{name} (temps)', build(),
                      Xb_train, yb_train, Xb_val, yb_val, batch=32)
    results_b[name] = score_test_b(model.predict(Xb_test, verbose=0))

pd.DataFrame(results_b).T

This code trains and evaluates several different forecasting models on the temperature dataset and presents the results in a Pandas DataFrame.

It defines a dictionary `contenders` that maps model names to lambda functions which create instances of those models using their respective build functions (defined previously). The models included are GRU, LSTM, vanilla transformer, PatchTST-lite, and linear regression. Note that all these models are configured for univariate forecasting (input channel size of 1) with parameters adjusted for the temperature dataset.

The code then iterates through each model in the `contenders` dictionary:

1.  **Model Creation & Training:** It sets the random seed using `keras.utils.set_random_seed(CFG.SEED)` to ensure reproducibility. Then, it calls the build function associated with the current model name to create an instance of the model. The model is trained using the `fit_timed` function, passing in the training data (`Xb_train`, `yb_train`), validation data (`Xb_val`, `yb_val`), and a descriptive name.
2.  **Evaluation:** It generates predictions on the test set using `model.predict()`. The performance of the trained model is then evaluated using the `score_test_b` function, which calculates RMSE on the temperature dataset. The resulting RMSE value is stored in the `results_b` dictionary with the model name as the key.

Finally, it creates a Pandas DataFrame from the `results_b` dictionary and transposes it (`.T`) to display the models as rows and the RMSE values as columns. This provides a clear comparison of the performance of different forecasting models on the temperature dataset.

In [ ]:
rematch = pd.DataFrame(results_b).T
rematch['params'] = [param_counts.get(f'{m} (temps)', 0) for m in rematch.index]
rematch['train_s'] = [train_times.get(f'{m} (temps)', 0.0) for m in rematch.index]
rematch.sort_values('RMSE')

This code enhances the results DataFrame by adding information about model complexity and training time, then sorts the results based on RMSE to identify the best-performing models.

First, it creates a Pandas DataFrame named `rematch` from the `results_b` dictionary (which contains the RMSE values for each model). The `.T` transposes the DataFrame so that models are rows and metrics are columns.

Next, it adds two new columns to the `rematch` DataFrame:

1.  **'params':** This column stores the number of trainable parameters for each model. It retrieves these values from the `param_counts` dictionary using the model name (e.g., 'GRU (temps)') as the key. If a model is not found in the dictionary, it defaults to 0.
2.  **'train\_s':** This column stores the training time (in seconds) for each model. It retrieves these values from the `train_times` dictionary using the model name as the key. If a model is not found, it defaults to 0.0.

Finally, it sorts the `rematch` DataFrame in ascending order based on the 'RMSE' column using `rematch.sort_values('RMSE')`. This arranges the models from best (lowest RMSE) to worst performance, making it easy to identify the most accurate model for this temperature forecasting task and assess its complexity and training time.

In [ ]:
plt.figure(figsize=(CFG.img_dim1 * 0.7, CFG.img_dim2))
learned = rematch[rematch['params'] > 0]
plt.scatter(learned['params'], learned['RMSE'], s=90, zorder=3)
for name, row in learned.iterrows():
    plt.annotate(f'  {name}', (row['params'], row['RMSE']), fontsize=11,
                 va='bottom')
plt.xscale('log')
plt.xlabel('trainable parameters (log scale)')
plt.ylabel('test RMSE (deg C)')
plt.title('The rematch in one picture: accuracy vs model size, Melbourne temperatures')
plt.show()

This code generates a scatter plot visualizing the relationship between model complexity (number of trainable parameters) and forecasting accuracy (RMSE) for the models that have trainable parameters.

It begins by creating a new figure with specified dimensions. Then, it filters the `rematch` DataFrame to include only models with more than 0 trainable parameters, storing the result in the `learned` variable. This excludes any models where parameter counts weren't available or were zero.

A scatter plot is created using `plt.scatter()`, plotting the number of trainable parameters (from the 'params' column) on the x-axis and the test RMSE (from the 'RMSE' column) on the y-axis. The marker size is set to 90, and `zorder=3` ensures that the markers are drawn on top of any annotations.

For each model in the `learned` DataFrame, an annotation is added using `plt.annotate()`. The annotation displays the model name next to its corresponding point on the scatter plot. The vertical alignment (`va='bottom'`) positions the text below the marker.

The x-axis scale is set to logarithmic using `plt.xscale('log')` to better visualize the wide range of parameter counts.  Appropriate labels are added for both axes ("trainable parameters (log scale)" and "test RMSE (deg C)"), and a title is given to the plot: "The rematch in one picture: accuracy vs model size, Melbourne temperatures".

Finally, the plot is displayed using `plt.show()`. This visualization allows for a quick assessment of whether more complex models (with more parameters) consistently achieve better forecasting accuracy or if there's a point of diminishing returns.

# Head-to-head: the full scoreboard

In [ ]:
scoreboard = pd.DataFrame(results).T
scoreboard['params'] = [param_counts.get(m, 0) for m in scoreboard.index]
scoreboard['train_s'] = [train_times.get(m, 0.0) for m in scoreboard.index]
scoreboard.sort_values('RMSE')

This code creates a comprehensive "scoreboard" DataFrame that consolidates the results from both forecasting tasks (item-level sales and temperature forecasting). It combines the performance metrics, model complexity, and training time for all models tested across both datasets.

First, it creates a Pandas DataFrame named `scoreboard` from the `results` dictionary. The `.T` transposes the DataFrame so that models are rows and metrics are columns. This `results` dictionary presumably contains the combined results from both the sales data experiments and the temperature data experiments.

Next, it adds two new columns to the `scoreboard` DataFrame:

1.  **'params':** This column stores the number of trainable parameters for each model. It retrieves these values from the `param_counts` dictionary using the model name (e.g., 'GRU') as the key. If a model is not found in the dictionary, it defaults to 0.
2.  **'train\_s':** This column stores the training time (in seconds) for each model. It retrieves these values from the `train_times` dictionary using the model name as the key. If a model is not found, it defaults to 0.0.

Finally, it sorts the `scoreboard` DataFrame in ascending order based on the 'RMSE' column using `scoreboard.sort_values('RMSE')`. This arranges the models from best (lowest RMSE) to worst performance across both datasets, providing an overall ranking of their forecasting capabilities. The resulting `scoreboard` DataFrame provides a complete overview of model performance, complexity, and training time for all experiments conducted in this notebook.

In [ ]:
plt.figure(figsize=(CFG.img_dim1 * 0.8, CFG.img_dim2))
ordered = scoreboard.sort_values('RMSE', ascending=False)
colors = ['firebrick' if p == 0 else ('seagreen' if 'ransformer' in m or 'Patch' in m
          else 'steelblue')
          for m, p in zip(ordered.index, ordered['params'])]
plt.barh(ordered.index, ordered['RMSE'], color=colors)
for i, (m, row) in enumerate(ordered.iterrows()):
    label = 'baseline' if row['params'] == 0 else f"{int(row['params']):,} params"
    plt.text(row['RMSE'] + 0.05, i, label, va='center', fontsize=10)
plt.xlabel('test RMSE (units sold) — lower is better')
plt.title('Store demand panel: every method of the episode')
plt.xlim(0, ordered['RMSE'].max() * 1.18)
plt.show()

This code generates a horizontal bar chart visualizing the performance of all forecasting models across both datasets (item-level sales and temperature), highlighting model complexity and categorizing models based on their architecture.

It begins by creating a new figure with specified dimensions. Then, it sorts the `scoreboard` DataFrame in descending order based on the 'RMSE' column using `scoreboard.sort_values('RMSE', ascending=False)`, storing the result in the `ordered` variable. This arranges the models from worst to best performance.

A list of colors is created for each model:

*   Models with 0 parameters (baselines) are colored 'firebrick'.
*   Models containing "transformer" or "Patch" in their name (Transformer-based architectures) are colored 'seagreen'.
*   All other models are colored 'steelblue'.

A horizontal bar chart is created using `plt.barh()`, plotting the RMSE values on the x-axis and model names on the y-axis. The bars are colored according to the defined color scheme.

For each model, a text label is added next to its corresponding bar using `plt.text()`. If the model has 0 parameters (a baseline), the label is "baseline". Otherwise, the label displays the number of trainable parameters formatted with commas for readability. The vertical alignment (`va='center'`) centers the text within the bar.

The x-axis is labeled "test RMSE (units sold) — lower is better", and the plot is titled "Store demand panel: every method of the episode".  The x-axis limits are set to extend slightly beyond the maximum RMSE value for better visualization.

Finally, the plot is displayed using `plt.show()`. This visualization provides a clear comparison of model performance across both datasets, highlighting the trade-offs between accuracy and complexity, and visually categorizing models based on their architecture.